# 4. Hafta Ödevi: Pandas ile Gerçek Veri Temizleme (Titanic)

**Menti:** Burçinnur Alkan  
**Konu:** Series, DataFrame, İndeksleme, Eksik Veriler, GroupBy  
**Haftalık Görev:** Ham bir veri setinin **eksik verilerini temizlemek**.

Bu ödevde Kaggle'ın klasik **Titanic** veri setini (891 yolcu) kullandım. Veriyi indirip `titanic.csv` olarak notebook'un yanına koydum. Gerçek veride 3 farklı sütunda eksik değer var ve her birini **farklı bir yöntemle** temizledim.

## 1. Veriyi Yükleme

`pd.read_csv` ile CSV dosyasını bir DataFrame'e okuyorum.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("titanic.csv")
print("Boyut (satir, sutun):", df.shape)
df.head()

Boyut (satir, sutun): (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
# Sutunlarin tipleri ve dolu deger sayilari
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


## 2. İndeksleme (`loc` ve `iloc`)

Temizlemeden önce veriyi biraz tanıyalım.

In [3]:
# iloc: konuma gore -> ilk yolcu
print("Ilk yolcu (iloc[0]):")
print(df.iloc[0][["Name", "Sex", "Age", "Survived"]])

# loc: kosula gore -> 60 yas ustu yolcular
print("\n60 yas ustu yolcular (loc):")
df.loc[df["Age"] > 60, ["Name", "Age", "Pclass", "Survived"]]

Ilk yolcu (iloc[0]):
Name        Braund, Mr. Owen Harris
Sex                            male
Age                            22.0
Survived                          0
Name: 0, dtype: object

60 yas ustu yolcular (loc):


,Name,Age,Pclass,Survived
33,"Wheadon, Mr. Edward H",66.0,2,0
54,"Ostby, Mr. Engelhart Cornelius",65.0,1,0
96,"Goldschmidt, Mr. George B",71.0,1,0
116,"Connors, Mr. Patrick",70.5,3,0
170,"Van der hoef, Mr. Wyckoff",61.0,1,0
252,"Stead, Mr. William Thomas",62.0,1,0
275,"Andrews, Miss. Kornelia Theodosia",63.0,1,1
280,"Duane, Mr. Frank",65.0,3,0
326,"Nysveen, Mr. Johan Hansen",61.0,3,0
438,"Fortune, Mr. Mark",64.0,1,0


## 3. ⭐ ASIL GÖREV: Eksik Verileri Bulma

Önce hangi sütunda kaç eksik değer var, onu görelim.

In [4]:
eksikler = df.isnull().sum()
print("Sutun basina eksik deger sayisi:")
print(eksikler[eksikler > 0])

# Yuzde olarak da bakalim
print("\nEksik oranlari (%):")
print((df.isnull().mean() * 100).round(1)[df.isnull().mean() > 0])

Sutun basina eksik deger sayisi:
Age         177
Cabin       687
Embarked      2
dtype: int64

Eksik oranlari (%):
Age         19.9
Cabin       77.1
Embarked     0.2
dtype: float64


**Durum tespiti:** 3 sütunda eksik var, her biri farklı strateji gerektiriyor:

| Sütun | Eksik sayısı | Tip | Stratejim |
|-------|-------------|-----|-----------|
| **Age** | 177 | Sayısal | **Medyan** ile doldur |
| **Embarked** | 2 | Kategorik | **En sık değer (mod)** ile doldur |
| **Cabin** | 687 (%77) | Metin | Çok fazla eksik → **sütunu sil** |

## 4. Temizleme

In [5]:
df_temiz = df.copy()   # orijinali bozmayalim diye kopya uzerinde calisiyorum

# --- 1) Age: sayisal sutun, medyan ile dolduruyorum ---
# Medyan sectim cunku asiri buyuk/kucuk yaslardan ortalamaya gore daha az etkilenir.
yas_medyan = df_temiz["Age"].median()
df_temiz["Age"] = df_temiz["Age"].fillna(yas_medyan)
print(f"Age eksikleri medyan ({yas_medyan}) ile dolduruldu.")

Age eksikleri medyan (28.0) ile dolduruldu.


In [6]:
# --- 2) Embarked: kategorik sutun, en sik gorulen deger (mod) ile dolduruyorum ---
# Sayisal olmadigi icin ortalama alinamaz; en mantiklisi en sik binilen limandir.
en_sik_liman = df_temiz["Embarked"].mode()[0]
df_temiz["Embarked"] = df_temiz["Embarked"].fillna(en_sik_liman)
print(f"Embarked eksikleri en sik deger ('{en_sik_liman}') ile dolduruldu.")

Embarked eksikleri en sik deger ('S') ile dolduruldu.


In [7]:
# --- 3) Cabin: %77'si eksik, doldurmak anlamsiz -> sutunu tamamen siliyorum ---
df_temiz = df_temiz.drop(columns=["Cabin"])
print("Cabin sutunu silindi (cok fazla eksik oldugu icin).")
print("Kalan sutunlar:", list(df_temiz.columns))

Cabin sutunu silindi (cok fazla eksik oldugu icin).
Kalan sutunlar: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Embarked']


In [8]:
# --- Kontrol: artik hic eksik veri kaldi mi? ---
print("Temizleme sonrasi eksik deger sayisi:")
print(df_temiz.isnull().sum())
print("\nToplam kalan eksik:", df_temiz.isnull().sum().sum())

Temizleme sonrasi eksik deger sayisi:
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

Toplam kalan eksik: 0


## 5. Gruplandırma (GroupBy) — Temiz Veriyle Analiz

Temizlenmiş veri üzerinde anlamlı sorular soralım.

In [9]:
# Cinsiyete gore hayatta kalma orani (Survived: 1=kurtuldu, 0=kurtulamadi)
print("Cinsiyete gore hayatta kalma orani:")
print((df_temiz.groupby("Sex")["Survived"].mean() * 100).round(1))

# Yolculuk sinifina gore hayatta kalma orani
print("\nYolcu sinifina gore hayatta kalma orani (%):")
print((df_temiz.groupby("Pclass")["Survived"].mean() * 100).round(1))

Cinsiyete gore hayatta kalma orani:
Sex
female    74.2
male      18.9
Name: Survived, dtype: float64

Yolcu sinifina gore hayatta kalma orani (%):
Pclass
1    63.0
2    47.3
3    24.2
Name: Survived, dtype: float64


In [10]:
# Sinif basina ortalama bilet ucreti ve yas
print("Sinif bazinda ortalamalar:")
df_temiz.groupby("Pclass")[["Age", "Fare"]].mean().round(1)

Sinif bazinda ortalamalar:


,Age,Fare
Pclass,,
1,36.8,84.2
2,29.8,20.7
3,25.9,13.7


## 6. Sonuç ve Yorum

**Temizleme özeti:**

| Sütun | Sorun | Çözümüm | Neden |
|-------|-------|---------|-------|
| Age | 177 eksik | Medyan ile doldurdum | Sayısal; medyan uç değerlere dayanıklı |
| Embarked | 2 eksik | Mod ile doldurdum | Kategorik; ortalama alınamaz |
| Cabin | %77 eksik | Sütunu sildim | Doldurulamayacak kadar çok eksik |

**📝 Yorumum:**
- Eksik veriyle başa çıkmanın **tek bir doğru yolu yok**; sütunun tipine ve eksik oranına göre karar vermek gerekiyor.
- **Az eksik + sayısal** → ortalama/medyan ile doldur. **Az eksik + kategorik** → mod ile doldur. **Çok eksik** → sütunu silmek daha mantıklı.
- GroupBy sonuçları veriyi doğruladı: kadınların ve 1. sınıf yolcuların hayatta kalma oranı belirgin şekilde yüksek. Bu, "kadınlar ve çocuklar önce" ve sınıf ayrımının gerçek etkisini gösteriyor.